In [ ]:
!pip install -U -q transformers sentence-transformers chromadb accelerate

In [ ]:
import json
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

#Carregar modelo de embedding
modelo_embedding = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

#Carregar modelo para gerar texto
nome_llm = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(nome_llm)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    nome_llm,
    dtype=torch.float16,
    device_map="auto",
)

print("Modelos carregados")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelos carregados


In [ ]:
#Carregar notícias
with open("noticias.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

print(f"{len(noticias)} notícias carregadas.")

#Preparar documentos
documentos = []
ids = []
metadados = []

for i, noticia in enumerate(noticias):
    documento = (
        f"{noticia['titulo']}\n"
        f"{noticia['resumo']}"
    )

    documentos.append(documento)
    ids.append(f"{noticia['ativo'].lower()}_{i}")
    metadados.append({
        "ativo": noticia["ativo"],
        "data": noticia["data"],
        "fonte": noticia["fonte"]
    })

30 notícias carregadas.


In [ ]:
#Gerar embeddings
#enconde() --> transformar textos em vetores/embeddings
#30 textos --> 30 vetores
#.tolist() --> converte de array Numpy para lista python, ficando uma lista que possui outras listas
embeddings = modelo_embedding.encode(documentos, normalize_embeddings=True).tolist()
print(f"{len(embeddings)} embeddings gerados.")

30 embeddings gerados.


In [ ]:
#Criar objeto cliente do ChromaDB
#Objeto através do qual o programa conversa com o ChromaDB
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="noticias_carteira")
except Exception:
    pass

#Criar collection
#colecao_noticias é o objeto através do qual  as notícias são adicionadas e consultadas
colecao_noticias = chroma_client.create_collection(
    name="noticias_carteira",
    metadata={"hnsw:space": "cosine"}
)

#Indexar no chromaDB
colecao_noticias.add(embeddings=embeddings, documents=documentos, ids=ids, metadatas=metadados)
print(f"{len(documentos)} notícias indexadas em 'noticias_carteira'.")

30 notícias indexadas em 'noticias_carteira'.


#Teste de Recuperação

In [ ]:
consulta = "dividendos recentes"

vetor_consulta = modelo_embedding.encode(
    [consulta],
    prompt_name="query",
    normalize_embeddings=True
).tolist()

for ativo in ["PETR4", "MXRF11", "CMIG4"]:
    resultado = colecao_noticias.query(
        query_embeddings=vetor_consulta,
        n_results=2,
        where={"ativo": ativo}, #Filtragem por ativo
        include=["documents", "distances"]
    )

    print(f"\n[{ativo}]")
    for documento, distancia in zip(resultado["documents"][0], resultado["distances"][0]):
        print(f"[dist={distancia:.3f}] {documento}")


[PETR4]
[dist=0.480] Petrobras aprova R$ 17,4 bi em proventos
Conselho aprovou R$ 17,4 bilhões em dividendos e JCP como antecipação da remuneração de 2026, equivalente a R$ 1,348 por ação, pagos em duas parcelas em novembro e dezembro de 2026.
[dist=0.511] Pagamento da 2ª parcela de dividendos de PETR4
Petrobras pagou a segunda parcela de proventos, no valor de R$ 0,35048636 por ação, referente à antecipação de junho de 2026.

[MXRF11]
[dist=0.468] MXRF11 divulga dividendos de junho
Fundo fixou R$ 0,10 por cota para junho de 2026, repetindo o valor dos dois meses anteriores. Pagamento em 14/07/2026, data-base 30/06/2026, yield mensal de aproximadamente 1,03% sobre cotação de R$ 9,75.
[dist=0.550] MXRF11 mantém dividendo em julho
Dividendo de R$ 0,10 por cota mantido para julho de 2026, pago em 14/08/2026, data-base 31/07/2026. Yield mensal de 1,04% sobre cotação de R$ 9,58.

[CMIG4]
[dist=0.413] Cemig propõe R$ 1,8 bi em dividendos adicionais
Cemig propôs pagamento adicional de R$ 1,8

In [ ]:
SYSTEM_PROMPT_CARTEIRA = (
    "Você é um assistente de análise de carteiras. Você recebe a posição do "
    "investidor em um ativo e notícias recentes sobre esse ativo. "
    "Sua tarefa é resumir, em até 5 frases, os fatores das notícias que podem "
    "afetar a performance do ativo, tanto pontos positivos quanto riscos. "
    "Use SOMENTE informações que estejam nas notícias fornecidas. "
    "Não invente números, datas ou eventos que não estejam no contexto. "
    "Nunca recomende comprar, vender ou manter o ativo, isso não é "
    "recomendação de investimento, apenas um resumo informativo. "
    "Se as notícias fornecidas não tiverem relação com o que foi perguntado, "
    "diga isso explicitamente."
)

def analisar_posicao(ativo, quantidade, preco_medio, preco_atual=None, n_noticias=4):
    query = f"notícias recentes que podem afetar a performance de {ativo}"
    vetor = modelo_embedding.encode(
        [query], prompt_name="query", normalize_embeddings=True
    ).tolist()
    busca = colecao_noticias.query(
        query_embeddings=vetor,
        n_results=n_noticias,
        where={"ativo": ativo},
        include=["documents", "distances"],
    )
    trechos = busca["documents"][0]
    distancias = busca["distances"][0]

    # Dados da posição
    valor_investido = quantidade * preco_medio
    linhas_posicao = [
        f"Ativo: {ativo}",
        f"Quantidade: {quantidade}",
        f"Preço médio: R$ {preco_medio:.2f}",
        f"Valor investido: R$ {valor_investido:.2f}",
    ]
    if preco_atual is not None:
        variacao_pct = (preco_atual / preco_medio - 1) * 100
        linhas_posicao.append(f"Preço atual: R$ {preco_atual:.2f}")
        linhas_posicao.append(f"Variação: {variacao_pct:+.2f}%")
    texto_posicao = "\n".join(linhas_posicao)

    print(f"--- Posição ---\n{texto_posicao}\n")
    print("--- Notícias recuperadas ---")
    for trecho, dist in zip(trechos, distancias):
        print(f"  [dist={dist:.3f}] {trecho}")
    print()

    # Monta o prompt de geração
    contexto_noticias = "\n".join(trechos)
    mensagens = [
        {"role": "system", "content": SYSTEM_PROMPT_CARTEIRA},
        {
            "role": "user",
            "content": (
                f"Posição do investidor:\n{texto_posicao}\n\n"
                f"Notícias:\n{contexto_noticias}\n\n"
                "Resuma os fatores relevantes dessas notícias para essa posição."
            ),
        },
    ]

    #Aplicar as tags de chat para o Qwen
    #add_generation_prompt=True --> adicionar gatilho para Qwen responder
    texto_prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )

    #tokenizer --> fatia o texto em tokens e substitui cada um por seu respectivo ID.
    #return_tensor="pt" --> define o formato de saída dos tensores. "pt" significa PyTorch
    #.to(modelo_llm.device) --> move os tensores gerados para a mesma memória (device) onde o LLM está rodando.
    entradas = tokenizer([texto_prompt], return_tensors="pt").to(modelo_llm.device)

    #with torch.no_grad() --> desativa calculo de gradientes do pytorch, economizando RAM da GPU e acelerando execução
    with torch.no_grad():
        #LLM gera o texto
        #**entradas --> desempacota os tensores gerados pelo tokenizer (os input_ids e attention_mask já posicionados na GPU)
        #max_new_tokens --> modelo para parar de escrever assim que atingir 300 novos tokens gerados
        #do_sample=False --> estratégia de decodificação como Greedy (Gulosa): o modelo sempre escolhe o token com maior probabilidade
        saida = modelo_llm.generate(**entradas, max_new_tokens=300, do_sample=False)

    #decode --> pega os ids dos tokens e transforma em português legível
    #slicing para manter apenas o que o modelo escreveu do zero
    #skip_special_tokens=True --> remove tokens de controle internos do modelo
    resposta = tokenizer.decode(
        saida[0][entradas.input_ids.shape[1]:], skip_special_tokens=True
    )

    print("--- ANÁLISE ---")
    print(resposta)
    print("\n[Isto é um resumo informativo gerado por IA, não é recomendação de investimento.]")
    return resposta

In [ ]:
analisar_posicao("PETR4", 10, 40, 50)

--- Posição ---
Ativo: PETR4
Quantidade: 10
Preço médio: R$ 40.00
Valor investido: R$ 400.00
Preço atual: R$ 50.00
Variação: +25.00%

--- Notícias recuperadas ---
  [dist=0.308] Cotação e múltiplos de PETR4
PETR4 cotada a R$ 42,31, com P/L de 5,07 e dividend yield de 8,4% nos últimos 12 meses, LPA de R$ 8,35. Máxima de R$ 49,78 e mínima de R$ 36,89 nos últimos 6 meses.
  [dist=0.466] Pagamento da 2ª parcela de dividendos de PETR4
Petrobras pagou a segunda parcela de proventos, no valor de R$ 0,35048636 por ação, referente à antecipação de junho de 2026.
  [dist=0.492] Pagamento da 1ª parcela de dividendos de PETR4
Petrobras pagou a primeira parcela de proventos aprovada anteriormente, no valor de R$ 0,35048636 por ação.
  [dist=0.575] BTG: Petrobras depende do Brent em 2026
BTG Pactual mantém recomendação de compra e preço-alvo de US$ 15 por ADR, mas alerta que a empresa está mais exposta ao preço do Brent e ao risco-país em 2026, chamando o ano de decisivo.

--- ANÁLISE ---
A cotação 

'A cotação de PETR4 subiu significativamente de R$ 40.00 para R$ 50.00, representando uma variação de +25%. O preço máximo alcançado foi R$ 49.78 e o mínimo foi R$ 36.89 nos últimos seis meses. As notícias indicam que a empresa está mais exposta ao preço do Brent e ao risco-país em 2026, o que pode afetar sua performance futura. Além disso, a empresa pagou duas parcelas de dividendos, aumentando a liquidez e a rentabilidade dos ações.'